In [1]:
import pandas as pd
import shutil
import os
import numpy as np
import matplotlib.pyplot as plt
import onekey_algo.custom.components as okcomp
from onekey_algo import get_param_in_cwd

plt.rcParams['figure.dpi'] = 300
sel_m = get_param_in_cwd('sel_model')
model_names = get_param_in_cwd('summary_models')
# 获取配置
task = get_param_in_cwd('task_column') or 'label'
bst_model = get_param_in_cwd('sel_model') or 'LR'
labelf = get_param_in_cwd('label_file')
group_info = get_param_in_cwd('dataset_column') or 'group'
figsize = (8, 6)
# 读取label文件。
labels = [task]
label_data_ = pd.read_csv(labelf)
label_data_['ID'] = label_data_['ID'].map(lambda x: os.path.splitext(x)[0])
label_data_ = label_data_[['ID', group_info, task]]
label_data_ = label_data_.dropna(axis=0)
all_res = []
ids = label_data_['ID']
print(label_data_.columns)
label_data = label_data_[['ID'] + labels]
label_data

Index(['ID', 'group', 'label'], dtype='object')


,ID,label
0,张伟东201511-202112,0.000
1,孙晓东202106-202311,0.000
2,王文强202010-202410,0.000
3,王爽202101-202209,0.000
4,崔忠升202109-202310,0.693
...,...,...
230,周涣博20230503-20240522,1.758
231,孟庆迁20230918-20250401,1.526
232,王绍峰20231222-20241219,1.224
233,富文强20210519-20230301,1.758


In [17]:
import pandas as pd
from onekey_algo.custom.components.comp1 import normalize_df, merge_results
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error, mean_absolute_error


def get_period(x):
    if x < 24:
        return '< 2 years'
    elif x < 48:
        return '2-4 years'
    else:
        return '>= 4 years'
label_data = pd.read_csv('data/clinical.csv')[['ID', 'Elapsed_time', 'label']]
label_data['Elapsed_time'] = label_data['Elapsed_time'].map(lambda x: get_period(x))

for sel_model in ['SVM', 'RandomForest', 'resnet50', 'densenet121', 'vgg19']:
    for subset in get_param_in_cwd('subsets'):
        for mn in  model_names:
            ALL_results = []
            try:
                r = pd.read_csv(f"./results/{mn}_{sel_model}_{subset}.csv")
        #         display(r)
                r.columns = ['ID', 'Prediction']
                r['Signature'] = mn
                r['Count'] = 1
                ALL_results.append(r)
                ALL_results = pd.concat(ALL_results, axis=0)
                ALL_results =pd.merge(ALL_results, label_data, on='ID', how='inner')
                ALL_results['MSE'] = (ALL_results['label'] - ALL_results['Prediction']) ** 2
                ALL_results['MAPE'] = np.abs(ALL_results['label'] - ALL_results['Prediction'])
            #     display(ALL_results)
                results = ALL_results.groupby(['Signature', 'Elapsed_time']).mean().drop('Prediction', axis=1)
                results['Count'] = ALL_results.groupby(['Signature', 'Elapsed_time']).sum()['Count']
                results.to_csv(f'results/subgroup_{mn}_{sel_model}_{subset}.csv', index=True)
                display(results)
            except:
                pass

Count  label    MSE   MAPE
Signature Elapsed_time                            
Clinical  2-4 years        41  0.747  0.518  0.614
          < 2 years        57  0.499  0.509  0.602
          >= 4 years       36  1.145  0.737  0.757

Count  label    MSE   MAPE
Signature Elapsed_time                            
DLElapsed 2-4 years        41  0.747  0.368  0.551
          < 2 years        57  0.499  0.419  0.590
          >= 4 years       36  1.145  0.939  0.858

Count  label    MSE   MAPE
Signature Elapsed_time                            
Combined  2-4 years        41  0.747  0.611  0.680
          < 2 years        57  0.499  0.636  0.711
          >= 4 years       36  1.145  0.838  0.829

Count  label    MSE   MAPE
Signature Elapsed_time                            
Clinical  2-4 years        23  0.852  0.392  0.539
          < 2 years        20  0.362  0.506  0.673
          >= 4 years       15  1.055  0.794  0.806

Count  label    MSE   MAPE
Signature Elapsed_time                            
DLElapsed 2-4 years        23  0.852  0.626  0.714
          < 2 years        20  0.362  0.569  0.723
          >= 4 years       15  1.055  1.029  0.977

...

Count  label    MSE   MAPE
Signature Elapsed_time                            
DL2D      2-4 years        23  0.852  0.703  0.793
          < 2 years        20  0.362  0.518  0.672
          >= 4 years       15  1.055  0.916  0.899

Count  label    MSE   MAPE
Signature Elapsed_time                            
DL2D      2-4 years        11  0.505  0.679  0.798
          < 2 years        31  0.854  0.793  0.771
          >= 4 years        1  0.000  1.153  1.074

Count  label    MSE   MAPE
Signature Elapsed_time                            
DL2D      2-4 years        41  0.747  0.602  0.701
          < 2 years        57  0.499  0.504  0.681
          >= 4 years       36  1.145  1.610  1.113

Count  label    MSE   MAPE
Signature Elapsed_time                            
DL2D      2-4 years        23  0.852  0.618  0.758
          < 2 years        20  0.362  0.548  0.716
          >= 4 years       15  1.055  1.128  1.014

Count  label    MSE   MAPE
Signature Elapsed_time                            
DL2D      2-4 years        11  0.505  0.585  0.740
          < 2 years        31  0.854  0.889  0.860
          >= 4 years        1  0.000  0.666  0.816